# **Linear Cryptanalysis**

## **MiniSPN**

### **Kiến trúc**

![miniSPN](miniSPN.svg)

In [ ]:
from toys.minispn import miniSPN

print(f'{miniSPN.block_size = }')
print(f'{miniSPN.num_rounds = }')
print(f'{miniSPN.sbox_size  = }')
print(f'{miniSPN.sbox = }')
print(f'{miniSPN.perm = }')

miniSPN.block_size = 48
miniSPN.num_rounds = 6
miniSPN.sbox_size  = 6
miniSPN.sbox = [42, 35, 11, 41, 30, 47, 26, 61, 18, 51, 49, 58, 1, 34, 7, 5, 6, 23, 9, 37, 12, 19, 14, 27, 16, 36, 62, 54, 46, 45, 57, 21, 53, 32, 33, 39, 59, 52, 43, 15, 4, 63, 38, 8, 13, 0, 55, 2, 40, 50, 25, 48, 44, 28, 3, 17, 56, 29, 20, 10, 60, 31, 22, 24]
miniSPN.perm = [0, 6, 12, 18, 24, 30, 36, 42, 1, 7, 13, 19, 25, 31, 37, 43, 2, 8, 14, 20, 26, 32, 38, 44, 3, 9, 15, 21, 27, 33, 39, 45, 4, 10, 16, 22, 28, 34, 40, 46, 5, 11, 17, 23, 29, 35, 41, 47]


### **Phân tích LAT**

- Đầu tiên ta xây dựng 1 bảng linear approximation table

In [ ]:
def make_lat(sbox):
    n = len(sbox)
    lat = [[0]*n for _ in range(n)]

    for x in range(n):
        for y in range(n):
            lat[x][y] = 32 - sum([((x&i)^(y&sbox[i])).bit_count() % 2 for i in range(n)]) 

    return lat

lat = make_lat(miniSPN.sbox)

for row in lat:
    print(' '.join(f'{x:>3}' for x in row))

 32   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
  0   4   0  -4   0 -12   4   0  -4  -4   0   0  -4   4   4  -4   2  -2  -2   2  -2  -6  -2   2  -2  -2  -2  -2   2  -6   6  -2   2   2  -2  -2  -2  -2  -2  -2   6  10  -2  10   2  -2  -2   2   4 -12   4   4  -4  -4   0   0   0  -4  -4   0   0   4   0  -4
  0   6   2   8  -4   2  -6   0   0   2  -2   0  12  -2  -2   0   0   6   6  -4   0  -2   2   0   0   2   2   4   0  -6   6   0  -4  -2   2  -4  -4  -2   6   0   4   2  -2   4   4   2  10   0  -4  -2   6   0   0  -6  -2   0   4   2   2  -8  -8  -2   2   0
  0   6   6  -4   0   6  -2   4  -4  -6   2  -8  -4   2   2   0   2   0   0  -2  -6   0   0 -10  -2  -4  -4   2  -2  -4  -4   2   2   0   0  -2   2   0   0  -2   6   4   4  -6   6  -4  -4  -6   4   2   2   0  -4   2  -6   0   0   6 

- Từ bảng lat ta chọn các phương trình sao cho nó có bias cao nhất

In [ ]:
from itertools import product

n = len(miniSPN.sbox)
flatten = [(x, y, lat[x][y]) for x, y in product(range(n), range(n))]
flatten.sort(key = lambda e: abs(e[-1]))

for x, y, c in flatten[-29:-1]:
    print(f'{x:06b} | {y:06b} | {c:3} | {c/n}')

000001 | 000101 | -12 | -0.1875
000001 | 110001 | -12 | -0.1875
000010 | 001100 |  12 | 0.1875
000100 | 011100 | -12 | -0.1875
000110 | 111110 | -12 | -0.1875
001000 | 100101 | -12 | -0.1875
001000 | 111001 |  12 | 0.1875
001110 | 010111 |  12 | 0.1875
010100 | 011011 |  12 | 0.1875
010101 | 011010 |  12 | 0.1875
010111 | 000111 | -12 | -0.1875
011110 | 000010 |  12 | 0.1875
011111 | 111110 | -12 | -0.1875
100000 | 101101 |  12 | 0.1875
100101 | 100111 |  12 | 0.1875
100110 | 100101 |  12 | 0.1875
100110 | 110111 |  12 | 0.1875
100111 | 110010 | -12 | -0.1875
101010 | 110001 |  12 | 0.1875
101111 | 111111 | -12 | -0.1875
110001 | 100010 | -12 | -0.1875
110100 | 011000 | -12 | -0.1875
111011 | 011000 | -12 | -0.1875
111100 | 110100 | -12 | -0.1875
111101 | 001110 |  12 | 0.1875
001111 | 111100 |  14 | 0.21875
111110 | 000111 | -14 | -0.21875
010110 | 100111 | -18 | -0.28125


- Tiếp tục ta lọc các phương trình có hamming weight thấp 2 và có có input hamming weight là 1

In [ ]:
hamm_2_filter = [e for e in flatten if e[0].bit_count() + e[1].bit_count() == 2]
input_hamm_1_filter  = [e for e in flatten if e[0].bit_count() == 1]
output_hamm_1_filter = [e for e in flatten if e[1].bit_count() == 1]


print('Relation has 2 hamming weight')
for x, y, c in hamm_2_filter[-7:-1]:
    print(f'{x:06b} | {y:06b} | {c:3} | {c/n}')

print('\nRelation input has 1 hamming weight')
for x, y, c in input_hamm_1_filter[-11:-1]:
    print(f'{x:06b} | {y:06b} | {c:3} | {c/n}')

print('\nRelation output has 1 hamming weight')
for x, y, c in output_hamm_1_filter[-9:-1]:
    print(f'{x:06b} | {y:06b} | {c:3} | {c/n}')

Relation has 2 hamming weight
001000 | 000001 |  -6 | -0.09375
010000 | 000001 |  -6 | -0.09375
010000 | 000010 |  -6 | -0.09375
010000 | 100000 |  -6 | -0.09375
100000 | 000010 |  -6 | -0.09375
000100 | 100000 |  -8 | -0.125

Relation input has 1 hamming weight
000010 | 101110 |  10 | 0.15625
000100 | 101001 |  10 | 0.15625
100000 | 000011 | -10 | -0.15625
100000 | 110101 | -10 | -0.15625
000001 | 000101 | -12 | -0.1875
000001 | 110001 | -12 | -0.1875
000010 | 001100 |  12 | 0.1875
000100 | 011100 | -12 | -0.1875
001000 | 100101 | -12 | -0.1875
001000 | 111001 |  12 | 0.1875

Relation output has 1 hamming weight
011001 | 010000 | -10 | -0.15625
100001 | 100000 |  10 | 0.15625
101000 | 100000 |  10 | 0.15625
101001 | 000001 |  10 | 0.15625
101100 | 010000 |  10 | 0.15625
110100 | 000100 |  10 | 0.15625
111010 | 010000 |  10 | 0.15625
111111 | 000010 |  10 | 0.15625


### **Xây dựng trail từ các phương trình có bias cao**

- Bước đầu chọn box đầu tiên với phương có output có hamming là 1 để cho nó tỏa ra càng ít sbox càng tốt (`011110 | 000010 |  12 | 0.1875`)
$$
x_1 + +x_2 + x_3 + x_4 + y_4 = 0
$$
- Tiếp tục sử dụng các relation có hamming 2 ở các box tiếp theo để giả thiểu tác động tới nhiều sbox.
- Và sbox cuối cùng sẽ toả ra với output có trọng số 3-4 kiểm soát vừa đủ sbox để bruteforce.

*Trail 1*
![miniSPN trail 1](miniSPN_T1.svg)

*Trait 2*
![miniSPN trail 2](miniSPN_T2.svg)

**Phân tích**

Trước tiên ta kí hiệu các round bằng mũ ví dụ input round r là $x^r$

*Các phương trình chung 2 trail*
$$
x_0^0 + x_5^0 + y_0^0 = 0, \text{bias} = \frac{10}{64}
$$
$$
x_3^1 + y_0^1 = 0, \text{bias} = \frac{-8}{64}
$$
$$
x_0^2 + y_4^2 = 0, \text{bias} = \frac{-6}{64}
$$
$$
x_0^3 + y_4^3 = 0, \text{bias} = \frac{-6}{64}
$$

*Phương trình cuối của trail 1*
$$
x_3^4 + y_1^4 + y_2^4 + y_3^4 = 0, \text{bias} = \frac{-12}{64}
$$

*Phương trình cuối của trail 2*
$$
x_3^4 + y_0^4 + y_2^4 + y_5^4 = 0, \text{bias} = \frac{10}{64}
$$

=> Áp dụng piling-up lemma và ghép các phương trình lại ta có 2 phương trình cuối với 2 bias tổng hợp là

$$
x_0^0 + x_5^0 + y_1^4 + y_2^4 + y_3^4 = 0, \text{bias} = 2^{5-1} \times \frac{10 \times -8 \times -6 \times -6 \times -12}{64^5}
$$
$$
x_0^0 + x_5^0 + y_0^4 + y_2^4 + y_5^4 = 0, \text{bias} = 2^{5-1} \times \frac{10 \times -8 \times -6 \times -6 \times 10}{64^5}
$$

In [ ]:
print('Bias of 1st equation: ', bias1st := 2**4*(10*-8*-6*-6*-12)/(64**5))
print('Bias of 2nd equation: ', bias2nd := 2**4*(10*-8*-6*-6*10)/(64**5))

print('Number of (plaintext, ciphertext) 1st equation need: ', round(8*bias1st**-2))
print('Number of (plaintext, ciphertext) 2st equation need: ', round(8*bias2nd**-2))

Bias of 1st equation:  0.000514984130859375
Bias of 2nd equation:  -0.0004291534423828125
Number of (plaintext, ciphertext) 1st equation need:  30164928
Number of (plaintext, ciphertext) 2st equation need:  43437496


### **Chiến thuật giải**

- Từ kết quả trên ta thu thập khoảng 33M pair cho trail 1
- Giải trail 1 và thu được 18 bits
- Vì trail 1 và trail 2 trùng ở 6 bit
- Tiếp tục dùng bộ pair trên, cố định 6 bit và giải để ra 12 bit còn lại (với xác xuất cao hơn và thời gian nhanh hơn)

In [19]:
from pwn import process, context

context.log_level = 'debug'
io = process(['python', 'oracle.py'], cwd='toys')

# print(io.recvall())

def choose_cipher():
    io.recvuntil(b'>>> ')
    io.sendline(b'1')

def encrypt_oracle(h):
    io.recvuntil(b'>>> ').decode()
    io.sendline(b'1')
    io.recvuntil(b'>>> ').decode()
    io.sendline(h.encode())
    return io.recvline().decode().split(': ')[-1][:-1]

choose_cipher()

context.log_level = 'info'

[x] Starting local process '/home/hikinx/miniforge3/envs/sage/bin/python' argv=[b'python', b'oracle.py'] 
[+] Starting local process '/home/hikinx/miniforge3/envs/sage/bin/python' argv=[b'python', b'oracle.py'] : pid 19034
[DEBUG] Received 0x3c bytes:
    b'Choose cipher for oracle:\n'
    b'  1. MiniSPN\n'
    b'  2. MiniFeistel\n'
    b'>>> '
[DEBUG] Sent 0x2 bytes:
    b'1\n'


### **Thu thập các cặp plaintext, ciphertext**

In [ ]:
import tqdm

ciphertext = []
B = 1000
for i in tqdm.trange(0, 33_000_000, B):
    inp = ''
    for j in range(i, i + B):
        plt  = f'{j:012x}'
        inp += f'1\n{plt}\n'
    io.send(inp.encode())
    for j in range(i, i + B):
        cpt = io.recvline_contains(b'Your encrypted block: ').decode().split(': ')[-1][:-1]
        ciphertext.append(int(cpt, 16))

print(ciphertext[:3])

  4%|▍         | 1378/33000 [08:08<6:34:07,  1.34it/s] 